# 01 — Ingest Contoso Data

Downloads Contoso V2 parquet dataset and writes Delta tables to the Lakehouse (bronze layer).

**Output tables:** `bronze_sales`, `bronze_customer`, `bronze_product`, `bronze_store`, `bronze_date`, `bronze_currency_exchange`

In [ ]:
# Install py7zr for 7z extraction
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'py7zr', '-q'],
                   capture_output=True, text=True)
import importlib; importlib.invalidate_caches()
print(f'py7zr install: exit={r.returncode}')

In [ ]:
# Download + extract + write all bronze tables in one cell
# (keeps /tmp files in same process scope)
import urllib.request, os, py7zr, pandas as pd

DOWNLOAD_URL = 'https://github.com/sql-bi/Contoso-Data-Generator-V2-data/releases/download/ready-to-use-data/parquet-100k.7z'
DOWNLOAD_PATH = '/tmp/contoso.7z'
EXTRACT_PATH = '/tmp/contoso/'

# Download
print('Downloading...')
urllib.request.urlretrieve(DOWNLOAD_URL, DOWNLOAD_PATH)
print(f'Downloaded: {os.path.getsize(DOWNLOAD_PATH):,} bytes')

# Extract
os.makedirs(EXTRACT_PATH, exist_ok=True)
with py7zr.SevenZipFile(DOWNLOAD_PATH, mode='r') as z:
    z.extractall(path=EXTRACT_PATH)
print('Extracted:', sorted(os.listdir(EXTRACT_PATH)))

# Table mapping: parquet filename -> Delta table name
TABLES = {
    'sales':           'bronze_sales',
    'customer':        'bronze_customer',
    'product':         'bronze_product',
    'store':           'bronze_store',
    'date':            'bronze_date',
    'currencyexchange':'bronze_currency_exchange',
}

# Read via pandas (avoids Spark ABFS auth issue with /tmp paths),
# convert to Spark DF, write as Delta (saveAsTable uses Lakehouse metastore)
for fname, table in TABLES.items():
    path = os.path.join(EXTRACT_PATH, f'{fname}.parquet')
    if not os.path.exists(path):
        print(f'SKIP {fname} - not found')
        continue
    pdf = pd.read_parquet(path)
    df = spark.createDataFrame(pdf)
    df.write.format('delta').mode('overwrite').saveAsTable(table)
    print(f'OK {table}: {len(pdf):,} rows')

print('\nIngestion complete!')

In [ ]:
# Verify row counts and inspect continent values
TABLES = ['bronze_sales', 'bronze_customer', 'bronze_product',
          'bronze_store', 'bronze_date', 'bronze_currency_exchange']
print('Row counts:')
for t in TABLES:
    try:
        print(f'  {t}: {spark.table(t).count():,}')
    except Exception as e:
        print(f'  {t}: ERROR {e}')

print('\nDistinct continent values in bronze_customer:')
spark.table('bronze_customer').select('continent').distinct().orderBy('continent').show()